<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/Parsing_PDFs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Parsing PDFs and Complex Documents

Every RAG pipeline in this course started from text that was already clean. Reality is rarely so polite: the documents worth indexing — papers, reports, invoices, manuals — arrive as **PDFs**, and a PDF is not a document. It is a set of *printing instructions*: glyphs at coordinates, columns that exist only visually, tables drawn with lines, sometimes just photographs of pages. Parsing is the act of turning that back into text a pipeline can chunk, embed, and cite — and it is where ingestion quality is won or lost.

This lesson climbs the three levels of PDF parsing:

1. **`pypdf`** — read the embedded text layer: instant, free, offline… and blind to layout
2. **Native file understanding** — hand the *file itself* to Gemini, OpenAI, or Anthropic: the model reads pages as pages (layout, tables, figures included)
3. **Managed parsing services** — when scans, volume, and gnarly tables justify a dedicated tool

📎 *A grounding fact before we start: the production AI Tutor needs none of this — it ingests Markdown and HTML sources directly. The best parser is the one you don't run. But the moment your corpus includes PDFs, this lesson is the difference between indexing knowledge and indexing noise.*

## 🧭 What You'll Learn

- What a PDF actually is, and why "extract the text" is a harder promise than it sounds
- **Level 1:** `pypdf` for born-digital PDFs — and a live demonstration of where the text layer betrays you (tables, columns, reading order)
- **Level 2:** native file understanding on **all three providers** — the same paper, three upload dialects, real table extraction
- **Slicing pages before upload** — the cheapest cost optimization in document AI
- **Parse → extract**: PDF in, validated Pydantic object out, in a single native call (📎 the structured-outputs lesson, applied to files)
- **Level 3:** an honest look at managed parsers (LlamaParse, Azure Document Intelligence, AWS Textract, open-source Docling & friends) — what they cost and when they earn it (as of July 2026)

## 1. Setup: Environment, Keys, and Providers

The standard course setup cell. This lesson shows native file understanding on **all three providers side by side**, so it asks for all three keys. If you only have some: edit `REQUIRED_KEYS` down to what you have, set `PROVIDER` to match, and skip the other providers' cells — Levels 1 and 3, and the schema-extraction demo on your selected provider, work regardless.

One extra pinned install: `pypdf`.

The **model field is an editable dropdown** (`{allow-input: true}`): pick one of the listed course defaults, or type any newer model ID straight into the box — no code changes needed. (Locally, simply edit the string.) For bulk work — many pages rather than one hard one — the dropdown also offers `gemini-3.5-flash-lite`, which Google positions as its high-throughput, low-cost option for document parsing and extraction.

In [1]:
# ============================================================
# ⚙️ Setup — environment, dependencies, API keys, provider
# ============================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules

# Pick your model provider (dropdown in Colab; edit the value locally)
PROVIDER = "gemini"  # @param ["gemini", "openai", "anthropic"]

# Pick a model for the selected provider — or TYPE any newer model ID into the
# box (the dropdown is editable thanks to allow-input):
CHAT_MODEL = "gemini-3.7-flash"  # @param ["gemini-3.7-flash", "gemini-3.5-flash-lite", "gpt-5.6-luna", "claude-sonnet-5"] {allow-input: true}

REQUIRED_KEYS = ["GOOGLE_API_KEY", "OPENAI_API_KEY", "ANTHROPIC_API_KEY"]

if IN_COLAB:
    import importlib
    import site
    import subprocess

    # Shared install profile, pinned course-wide (July 2026). Library updates can
    # change behavior, so we pin versions to keep every cell reproducible.
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "-U",
            "google-genai==2.3.0",
            "openai==2.46.0",
            "anthropic==0.117.0",
            "pypdf==6.10.2",
        ],
        check=True,
    )
    importlib.reload(site)  # make newly installed packages importable without a runtime restart

    # In Colab: Secrets tab (🔑 icon in the left sidebar) → Add new secret →
    # name it e.g. GOOGLE_API_KEY, paste the key, and toggle notebook access on.
    from google.colab import userdata

    for key in REQUIRED_KEYS:
        os.environ[key] = userdata.get(key)

if not IN_COLAB:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

    # Locally: install dependencies once from the repo's requirements file.
    # API keys live in a .env file at the repo root (never hardcode keys in cells).
    from dotenv import load_dotenv

    load_dotenv()
    missing = [k for k in REQUIRED_KEYS if not os.getenv(k)]
    assert not missing, f"Missing from .env: {missing}"

print(f"✅ Setup complete — {'Colab' if IN_COLAB else 'local'} | provider: {PROVIDER}")

✅ Setup complete — local | provider: gemini


## 2. Course Models per Provider

No `generate()` in this lesson — file inputs need provider-specific request shapes, which is precisely what we are here to learn. We keep only the model table (with the usual rule: your setup-cell pick overrides the default).

In [2]:
# Course-standard default models per provider (August 2026)
MODELS = {
    "gemini": "gemini-3.7-flash",
    "openai": "gpt-5.6-luna",
    "anthropic": "claude-sonnet-5",
}

# The setup-cell form selection (or any typed model ID) overrides the default:
MODELS[PROVIDER] = CHAT_MODEL
print(f"Default provider for single-provider cells: {PROVIDER} → {MODELS[PROVIDER]}")

Default provider for single-provider cells: gemini → gemini-3.7-flash


## 3. Get Some Real PDFs

Three research papers from the course dataset — including the LoRA paper, whose benchmark tables will do the honest work of breaking Level 1 for us.

In [3]:
import io
import pathlib
import zipfile

import requests

# Course dataset — hosted in the Towards AI org dataset repo on Hugging Face
ZIP_URL = "https://huggingface.co/datasets/towardsai-tutors/full-stack-ai-engineering-data/resolve/main/parsing_PDFs_research_papers.zip"
PDF_DIR = pathlib.Path("research_papers_llamaparse")

if not PDF_DIR.exists():
    zipfile.ZipFile(io.BytesIO(requests.get(ZIP_URL, timeout=120).content)).extractall(".")

pdf_paths = sorted(PDF_DIR.glob("*.pdf"))
for p in pdf_paths:
    print(f"{p.name:>22} — {p.stat().st_size / 1e6:.1f} MB")

LORA_PDF = PDF_DIR / "2106.09685v2.pdf"  # "LoRA: Low-Rank Adaptation of Large Language Models"

      2106.09685v2.pdf — 1.6 MB
      2404.19756v2.pdf — 12.7 MB
      2405.07437v2.pdf — 0.6 MB


## 4. Level 1 — `pypdf`: Reading the Text Layer

A born-digital PDF (one exported from LaTeX, Word, a browser…) carries an embedded **text layer**, and [`pypdf`](https://pypdf.readthedocs.io/) reads it directly — no API, no cost, milliseconds per page. For clean single-column prose, this is all you need, which is why it is Level 1 and not a footnote.

In [4]:
import time

from pypdf import PdfReader

t0 = time.perf_counter()
reader = PdfReader(LORA_PDF)
full_text = "\n".join(page.extract_text() or "" for page in reader.pages)
elapsed = time.perf_counter() - t0

print(f"{LORA_PDF.name}: {len(reader.pages)} pages → {len(full_text):,} characters in {elapsed:.2f}s (free, offline)\n")
print(full_text[:400], "…")

2106.09685v2.pdf: 26 pages → 82,714 characters in 2.83s (free, offline)

LORA: L OW-R ANK ADAPTATION OF LARGE LAN-
GUAGE MODELS
Edward Hu∗ Yelong Shen∗ Phillip Wallis Zeyuan Allen-Zhu
Yuanzhi Li Shean Wang Lu Wang Weizhu Chen
Microsoft Corporation
{edwardhu, yeshe, phwallis, zeyuana,
yuanzhil, swang, luw, wzchen }@microsoft.com
yuanzhil@andrew.cmu.edu
(Version 2)
ABSTRACT
An important paradigm of natural language processing consists of large-scale pre-
training on gene …


In [5]:
# Now the betrayal. This paper's results live in TABLES — here is what the
# text layer makes of one. Find a table caption and read the "table" behind it:
anchor = full_text.find("Table 2")
print(full_text[anchor : anchor + 900])

Table 2: RoBERTabase, RoBERTalarge, and DeBERTaXXL with different adaptation methods on the
GLUE benchmark. We report the overall (matched and mismatched) accuracy for MNLI, Matthew’s
correlation for CoLA, Pearson correlation for STS-B, and accuracy for other tasks. Higher is better
for all metrics. * indicates numbers published in prior works.† indicates runs conﬁgured in a setup
similar to Houlsby et al. (2019) for a fair comparison.
Bias-only or BitFit is a baseline where we only train the bias vectors while freezing everything else.
Contemporarily, this baseline has also been studied by BitFit (Zaken et al., 2021).
Preﬁx-embedding tuning (PreEmbed) inserts special tokens among the input tokens. These spe-
cial tokens have trainable word embeddings and are generally not in the model’s vocabulary. Where
to place such tokens can have an impact on performance. We focus on “preﬁxing”, whi


**What just happened?** The abstract came out flawlessly — and the table came out as **word soup**: model names, dashes, and benchmark numbers in an order set by the PDF's internal drawing sequence, not by rows and columns. Nothing associates `87.3` with the model and metric it belongs to; chunk this and your index will happily "know" wrong numbers. Two more failure modes to keep in mind: multi-column papers can interleave columns into nonsense, and **scanned PDFs have no text layer at all** — `pypdf` returns empty strings, because there is nothing to read and it does no OCR.

The text layer tells you *what characters exist*. It cannot tell you *what the page means*. For that, something has to actually look at the page.

## 5. Level 2 — Native File Understanding: Send the File, Not Your Guess

All three course providers now accept PDFs **as first-class input**: the model sees each page the way you do — rendered, with layout, tables, and figures — alongside the text (as of July 2026; size caps apply, roughly 20–50 MB per request depending on provider, with file-upload APIs for more). This is a provider-specific feature, so per course custom we show **all three native dialects**.

First, the cheapest optimization in document AI: **don't upload pages you don't need**. Model reading is priced per token and a page costs on the order of a few thousand input tokens — so we use `pypdf` (Level 1 earning its keep) to find the pages that contain Table 2 and slice them into a mini-PDF:

In [6]:
from pypdf import PdfWriter


def slice_pdf(src_path, page_indices, out_path):
    """Write a new PDF containing only the given (0-based) pages."""
    writer = PdfWriter()
    reader = PdfReader(src_path)
    for i in page_indices:
        writer.add_page(reader.pages[i])
    with open(out_path, "wb") as f:
        writer.write(f)
    return pathlib.Path(out_path)


# Find the pages that mention Table 2 (pypdf as a page *locator* — a job it's great at)
table_pages = [i for i, page in enumerate(reader.pages) if "Table 2" in (page.extract_text() or "")]
print("Pages mentioning Table 2:", [p + 1 for p in table_pages])

TABLE_SLICE = slice_pdf(LORA_PDF, table_pages[:2] or [4, 5], "lora_table2_slice.pdf")
print(f"{TABLE_SLICE} — {TABLE_SLICE.stat().st_size / 1e3:.0f} KB "
      f"(vs {LORA_PDF.stat().st_size / 1e6:.1f} MB for the full paper)")

TASK = ("Extract Table 2 from this paper excerpt as a GitHub-flavored Markdown table, "
        "preserving every row, column header, and value exactly. "
        "Then add ONE sentence stating what the table shows.")

Pages mentioning Table 2: [6, 7]
lora_table2_slice.pdf — 146 KB (vs 1.6 MB for the full paper)


### 5.1 Gemini — `Part.from_bytes(..., mime_type="application/pdf")`

Inline bytes work up to ~20 MB per request; beyond that, upload once with `client.files.upload(...)` and reference the returned handle. *Runs if `GOOGLE_API_KEY` is set.*

In [7]:
if os.getenv("GOOGLE_API_KEY"):
    from google import genai
    from google.genai import types as gtypes

    g_client = genai.Client()
    response = g_client.models.generate_content(
        model=MODELS["gemini"],
        contents=[
            gtypes.Part.from_bytes(data=TABLE_SLICE.read_bytes(), mime_type="application/pdf"),
            TASK,
        ],
    )
    print(response.text)
else:
    print("GOOGLE_API_KEY not set — skipping the Gemini version (that's fine).")

### Table 2

| Model & Method | # Trainable Parameters | MNLI | SST-2 | MRPC | CoLA | QNLI | QQP | RTE | STS-B | Avg. |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| $\text{RoB}_{\text{base}}\text{ (FT)}^*$ | 125.0M | 87.6 | 94.8 | 90.2 | 63.6 | 92.8 | 91.9 | 78.7 | 91.2 | 86.4 |
| $\text{RoB}_{\text{base}}\text{ (BitFit)}^*$ | 0.1M | 84.7 | 93.7 | 92.7 | 62.0 | 91.8 | 84.0 | 81.5 | 90.8 | 85.2 |
| $\text{RoB}_{\text{base}}\text{ (Adpt}^\text{D}\text{)}^*$ | 0.3M | 87.1±.0 | 94.2±.1 | 88.5±1.1 | 60.8±.4 | 93.1±.1 | 90.2±.0 | 71.5±2.7 | 89.7±.3 | 84.4 |
| $\text{RoB}_{\text{base}}\text{ (Adpt}^\text{D}\text{)}^*$ | 0.9M | 87.3±.1 | 94.7±.3 | 88.4±.1 | 62.6±.9 | 93.0±.2 | 90.6±.0 | 75.9±2.2 | 90.3±.1 | 85.4 |
| $\text{RoB}_{\text{base}}\text{ (LoRA)}$ | 0.3M | 87.5±.3 | 95.1±.2 | 89.7±.7 | 63.4±1.2 | 93.3±.3 | 90.8±.1 | 86.6±.7 | 91.5±.2 | 87.2 |
| $\text{RoB}_{\text{large}}\text{ (FT)}^*$ | 355.0M | 90.2 | 96.4 | 90.9 | 68.0 | 94.7 | 92.2 | 86.6 | 92.4

### 5.2 OpenAI — `input_file` with a base64 data URL

The Responses API takes files as `input_file` content parts — base64-encoded inline (as here), or uploaded once via `client.files.create(...)` and referenced by `file_id`.

In [8]:
if os.getenv("OPENAI_API_KEY"):
    import base64

    from openai import OpenAI

    o_client = OpenAI()
    b64 = base64.b64encode(TABLE_SLICE.read_bytes()).decode("utf-8")
    response = o_client.responses.create(
        model=MODELS["openai"],
        reasoning={"effort": "none"},
        input=[{
            "role": "user",
            "content": [
                {"type": "input_file", "filename": TABLE_SLICE.name,
                 "file_data": f"data:application/pdf;base64,{b64}"},
                {"type": "input_text", "text": TASK},
            ],
        }],
    )
    print(response.output_text)
else:
    print("OPENAI_API_KEY not set — skipping the OpenAI version (that's fine).")

| Model & Method | # Trainable Parameters | MNLI | SST-2 | MRPC | CoLA | QNLI | QQP | RTE | STS-B | Avg. |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| RoBbase (FT)* | 125.0M | 87.6 | 94.8 | 90.2 | 63.6 | 92.8 | 91.9 | 78.7 | 91.2 | 86.4 |
| RoBbase (BitFit)* | 0.1M | 84.7 | 93.7 | 92.7 | 62.0 | 91.8 | 84.0 | 81.5 | 90.8 | 85.2 |
| RoBbase (AdptD<br>)* | 0.3M | 87.1±.0 | 94.2±.1 | 88.5±1.1 | 60.8±.4 | 93.1±.1 | 90.2±.0 | 71.5±2.7 | 89.7±.3 | 84.4 |
| RoBbase (AdptD<br>)* | 0.9M | 87.3±.1 | 94.7±.3 | 88.4±.1 | 62.6±.9 | 93.0±.2 | 90.6±.0 | 75.9±2.2 | 90.3±.1 | 85.4 |
| RoBbase (LoRA) | 0.3M | 87.5±.3 | 95.1±.2 | 89.7±.7 | 63.4±1.2 | 93.3±.3 | 90.8±.1 | 86.6±.7 | 91.5±.2 | 87.2 |
| RoBlarge (FT)* | 355.0M | 90.2 | 96.4 | 90.9 | 68.0 | 94.7 | 92.2 | 86.6 | 92.4 | 88.9 |
| RoBlarge (LoRA) | 0.8M | 90.6±.2 | 96.2±.5 | 90.9±1.2 | 68.2±1.9 | 94.9±.3 | 91.6±.1 | 87.4±2.5 | 92.6±.2 | 89.0 |
| RoBlarge (AdptP<br>)† | 3.0M | 90.2±.3 | 96.1±.3 | 90.2±.7 | 68.3±1.0 | 94.8±.2 | 91.9±.1 

### 5.3 Anthropic — a `document` content block

Claude takes PDFs as `document` blocks (base64 inline as here, up to 32 MB per request; or upload via the Files API and reference by `file_id`).

The cell below is **commented out** — uncomment it to try it out. It runs only when `PROVIDER` is set to `"anthropic"` in the setup cell *and* `ANTHROPIC_API_KEY` is available.


In [ ]:
# 💡 TRY IT OUT — uncomment the block below to run the Anthropic version.
if PROVIDER == "anthropic" and os.getenv("ANTHROPIC_API_KEY"):
    pass
    # import base64
    #
    # from anthropic import Anthropic
    #
    # a_client = Anthropic()
    # b64 = base64.b64encode(TABLE_SLICE.read_bytes()).decode("utf-8")
    # response = a_client.messages.create(
    #     model=MODELS["anthropic"],
    #     max_tokens=2048,
    #     messages=[{
    #         "role": "user",
    #         "content": [
    #             {"type": "document",
    #              "source": {"type": "base64", "media_type": "application/pdf", "data": b64}},
    #             {"type": "text", "text": TASK},
    #         ],
    #     }],
    # )
    # print(response.content[0].text)
else:
    print("PROVIDER is not 'anthropic' (or ANTHROPIC_API_KEY is unset) — "
          "skipping the Anthropic version (that's fine).")


**What just happened?** The same two pages that `pypdf` turned into word soup came back as a **real Markdown table** — headers attached to columns, values attached to rows (**verify a few numbers against the paper**: that spot-check habit is part of the craft, because vision parsing is excellent but not infallible, and hallucinated table cells are the failure mode to fear). Three dialects, one idea:

| | Gemini | OpenAI | Anthropic |
|---|---|---|---|
| File goes in as | `Part.from_bytes(mime_type="application/pdf")` | `input_file` + base64 data URL | `document` block, base64 source |
| Bigger files | `client.files.upload(...)` | `client.files.create(...)` + `file_id` | Files API + `file_id` |

And the economics: we paid for ~2 pages, not 26, because a dumb-but-free tool located the pages first. **Level 1 and Level 2 are not rivals — they are a pipeline.**

## 6. Parse → Extract into a Schema (One Call)

📎 *This is the structured-outputs lesson pointed at a file: the PDF goes in, and a validated Pydantic object comes out — parsing and typed extraction in a single native call on your selected provider.* The mini-PDF here includes the title page plus the table pages, so the model can fill bibliographic fields and read results.

In [10]:
from pydantic import BaseModel, Field


class PaperCard(BaseModel):
    """A structured index card for one research paper."""

    title: str
    arxiv_id: str | None = Field(description="arXiv identifier, e.g. '2106.09685', or null if absent.")
    authors: list[str] = Field(description="Author names, in order.")
    one_line_summary: str = Field(description="What the paper shows, in one sentence.")
    table_takeaway: str = Field(description="One concrete finding from the table in this excerpt, with numbers.")


CARD_SLICE = slice_pdf(LORA_PDF, sorted({0, 1, *table_pages[:2]}), "lora_card_slice.pdf")
CARD_TASK = "Fill the paper card strictly from this paper excerpt."


def paper_card(pdf_path):
    """PDF in → validated PaperCard out, natively, on the selected PROVIDER."""
    data = pathlib.Path(pdf_path).read_bytes()

    if PROVIDER == "gemini":
        from google import genai
        from google.genai import types as gtypes

        client = genai.Client()
        response = client.models.generate_content(
            model=MODELS["gemini"],
            contents=[gtypes.Part.from_bytes(data=data, mime_type="application/pdf"), CARD_TASK],
            config=gtypes.GenerateContentConfig(
                response_mime_type="application/json", response_schema=PaperCard),
        )
        return response.parsed or PaperCard.model_validate_json(response.text)

    if PROVIDER == "openai":
        import base64

        from openai import OpenAI

        client = OpenAI()
        response = client.responses.parse(
            model=MODELS["openai"],
            reasoning={"effort": "none"},
            text_format=PaperCard,
            input=[{
                "role": "user",
                "content": [
                    {"type": "input_file", "filename": pathlib.Path(pdf_path).name,
                     "file_data": "data:application/pdf;base64," + base64.b64encode(data).decode("utf-8")},
                    {"type": "input_text", "text": CARD_TASK},
                ],
            }],
        )
        return response.output_parsed

    if PROVIDER == "anthropic":
        import base64

        from anthropic import Anthropic

        client = Anthropic()
        tool = {"name": "record_paper_card",
                "description": "Record the structured paper card.",
                "input_schema": PaperCard.model_json_schema()}
        response = client.messages.create(
            model=MODELS["anthropic"],
            max_tokens=2048,
            tools=[tool],
            tool_choice={"type": "tool", "name": "record_paper_card"},
            messages=[{
                "role": "user",
                "content": [
                    {"type": "document",
                     "source": {"type": "base64", "media_type": "application/pdf",
                                "data": base64.b64encode(data).decode("utf-8")}},
                    {"type": "text", "text": CARD_TASK},
                ],
            }],
        )
        block = next(b for b in response.content if b.type == "tool_use")
        return PaperCard.model_validate(block.input)

    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")


card = paper_card(CARD_SLICE)
print(card.model_dump_json(indent=2))

{
  "title": "LoRA: Low-Rank Adaptation of Large Language Models",
  "arxiv_id": "2106.09685",
  "authors": [
    "Edward Hu",
    "Yelong Shen",
    "Phillip Wallis",
    "Zeyuan Allen-Zhu",
    "Yuanzhi Li",
    "Shean Wang",
    "Lu Wang",
    "Weizhu Chen"
  ],
  "one_line_summary": "LoRA adapts large pre-trained language models to downstream tasks by freezing original weights and injecting trainable low-rank decomposition matrices, matching or exceeding full fine-tuning quality with drastically fewer parameters and no additional inference latency.",
  "table_takeaway": "On the GLUE benchmark (Table 2), RoBERTa base with LoRA achieves an overall average score of 87.2 using only 0.3M trainable parameters, outperforming full fine-tuning at 86.4 with 125.0M parameters."
}


**What just happened?** One request carried the file *and* the schema; back came a `PaperCard` with a checked `arxiv_id`, a typed author list, and a numbers-bearing takeaway — no text-extraction step, no parsing code, no fences. This "file → typed record" move is the modern default for **low-volume, high-value** documents: contracts, reports, papers, invoices in the hundreds. Which raises the obvious question — what about the millions?

## 7. Level 3 — When Managed Parsers Earn Their Keep

An honest decision guide. Dedicated parsing services — **LlamaParse** (LlamaCloud), **Azure Document Intelligence**, **AWS Textract**, and open-source pipelines like **Docling**, **unstructured**, and **marker** — exist because three workloads stay hard:

- **Scans and OCR at volume** — Level 2 reads scanned pages too, but paying model tokens to OCR a million-page archive is the expensive way; OCR-first pipelines and per-page parsers are built for it.
- **Gnarly tables and forms** — financial filings, nested spreadsheets-in-PDFs, invoices with checkboxes and handwriting. Specialized layout models plus human-verifiable intermediate output (Markdown/JSON per page) beat one-shot extraction when errors carry money.
- **Parse once, reuse forever** — a parser converts the document into Markdown/JSON you *own and cache*. Native file understanding re-reads (and re-bills) the pages on every call; a parsed artifact is paid for once and re-chunked for free every time you improve your pipeline.

**What it costs (as of July 2026):** LlamaParse prices per page in credits, by preset — 1 credit/page (Fast), 3 (Cost-effective), 10 (Agentic), 45 (Agentic Plus) — with extra credits at ~$1.25 per 1,000. So a clean-digital thousand-page batch parses for about a dollar on Fast, while the agentic presets that chew through brutal layouts run 10–45× that — always check the current pricing page before you commit a corpus, because these plans have already changed shape twice. Azure and AWS price per page in the same order of magnitude; the open-source options cost you GPUs and patience instead.

**The decision in one breath:** born-digital prose → `pypdf`; layout and tables matter on a bounded set → native file understanding (slice pages first); scans, volume, forms, or auditable per-page output → a managed or self-hosted parser. And if you control the source format — as our production tutor does by ingesting Markdown/HTML — **skip the whole problem**: the best parser is the one you never run.

In [11]:
# 🔬 OPTIONAL EXPERIMENT — bring your own gnarliest PDF
# Drop a PDF next to this notebook (drag it into Colab's file pane, or use any local
# path) and compare Level 1 vs Level 2 on the page that scares you most.
MY_PDF = ""  # e.g. "my_bank_statement.pdf"

if MY_PDF:
    my_reader = PdfReader(MY_PDF)
    print("— Level 1 (pypdf), first 600 chars of page 1 —")
    print((my_reader.pages[0].extract_text() or "(no text layer — a scan?)")[:600])

    my_slice = slice_pdf(MY_PDF, [0], "my_first_page.pdf")
    data = my_slice.read_bytes()
    TRANSCRIBE = "Transcribe this page faithfully as Markdown — tables as Markdown tables."
    print("\n— Level 2 (native, on your selected provider) —")

    if PROVIDER == "gemini":
        from google import genai
        from google.genai import types as gtypes

        result = genai.Client().models.generate_content(
            model=MODELS["gemini"],
            contents=[gtypes.Part.from_bytes(data=data, mime_type="application/pdf"), TRANSCRIBE],
        ).text
    elif PROVIDER == "openai":
        import base64

        from openai import OpenAI

        result = OpenAI().responses.create(
            model=MODELS["openai"], reasoning={"effort": "none"},
            input=[{"role": "user", "content": [
                {"type": "input_file", "filename": my_slice.name,
                 "file_data": "data:application/pdf;base64," + base64.b64encode(data).decode("utf-8")},
                {"type": "input_text", "text": TRANSCRIBE},
            ]}],
        ).output_text
    else:
        import base64

        from anthropic import Anthropic

        result = Anthropic().messages.create(
            model=MODELS["anthropic"], max_tokens=2048,
            messages=[{"role": "user", "content": [
                {"type": "document", "source": {"type": "base64", "media_type": "application/pdf",
                                                "data": base64.b64encode(data).decode("utf-8")}},
                {"type": "text", "text": TRANSCRIBE},
            ]}],
        ).content[0].text

    print(result[:1500])
else:
    print("Set MY_PDF to a PDF path to run this experiment.")

Set MY_PDF to a PDF path to run this experiment.


## 🔑 Key Takeaways

- A PDF is printing instructions, not a document — "just extract the text" is a promise the format never made.
- **Level 1 (`pypdf`)**: free, instant, offline; perfect for born-digital prose, useless for scans, dangerous for tables (word soup that indexes as wrong facts).
- **Level 2 (native file understanding)**: all three providers read PDFs as rendered pages — layout, tables, figures. Slice the pages you need first (`PdfWriter`): dumb-free tools locate, expensive-smart models read.
- **Parse → extract composes into one call**: file plus Pydantic schema in, validated object out — the structured-outputs lesson, applied to documents.
- **Level 3 (managed parsers)** wins on scans, volume, forms, and parse-once-reuse-forever economics — LlamaParse (credit-priced per page, ~$1.25/1k credits, as of July 2026), Azure Document Intelligence, AWS Textract, or open-source Docling/unstructured/marker.
- Verify extracted numbers against the source page — vision parsing is superb and still not an oath. And when you control the source format, ingest Markdown/HTML and skip parsing entirely, like the production tutor does.